# AIMedicine: разбор трёх случаев

Синтетическая когорта взрослых пациентов с внебольничной пневмонией. Клинические рекомендации — редакция 2024 года.

Карточка пациента → поиск по документу → план → сценарная динамика дня 3 → пересмотр → независимый аудит.

Notebook показывает сохранённые результаты из `reports/`: завершённый live-прогон с Mistral либо, если его нет, явно обозначенный offline-прогон. Показатели симуляции задаются сценарием; они используются для проверки переходов между этапами.


In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "aimedicine").is_dir() and (p / "reports").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Откройте notebook из корня проекта или папки notebooks")
def load(path):
    return json.loads(path.read_text(encoding="utf-8"))

expected_ids = {p["id"] for p in load(ROOT / "data/patients.json")}

def completed_run(mode):
    folder = ROOT / "reports" / mode
    manifest = folder / "summary.json"
    if not manifest.exists():
        return None, "нет завершающего summary.json"
    status_file = folder / "run_status.json"
    if status_file.exists() and load(status_file).get("status") != "complete":
        return None, "run_status.json не подтверждает завершение запуска"
    summary = load(manifest)
    if summary.get("mode") != mode or summary.get("status") in {"failed", "error"}:
        return None, "режим/статус манифеста не подтверждает успешный запуск"
    if {c.get("patient_id") for c in summary.get("cases", [])} != expected_ids or len(summary["cases"]) != len(expected_ids):
        return None, "манифест не содержит полный набор трёх пациентов"
    error_file = folder / "run_error.json"
    if error_file.exists() and error_file.stat().st_mtime_ns > manifest.stat().st_mtime_ns:
        return None, "после манифеста зафиксирована ошибка запуска"
    result = []
    for row in summary["cases"]:
        report_path = (folder / row["path"]).resolve()
        if report_path.parent != folder.resolve() or not report_path.is_file():
            return None, "отсутствует файл, указанный в манифесте"
        if report_path.stat().st_mtime_ns > manifest.stat().st_mtime_ns:
            return None, "файлы менялись после манифеста; возможен незавершённый повторный запуск"
        item = load(report_path)
        if item.get("mode") != mode or item.get("patient", {}).get("id") != row["patient_id"] or not item.get("plans"):
            return None, "история не соответствует манифесту"
        result.append((report_path, item))
    return result, "полный набор подтверждён summary.json"

traces, live_status = completed_run("live")
if traces is not None:
    selected_mode = "live"
    display(Markdown("Выбран **live**: найден полный завершённый запуск с API."))
else:
    traces, offline_status = completed_run("offline")
    selected_mode = "offline"
    display(Markdown(f"Выбран **offline**. Live не подтверждён: {live_status}. Offline не заменяет проверку LLM."))
    if traces is None:
        raise RuntimeError("Нет полного offline-запуска: " + offline_status + ". Выполните python -m aimedicine.cli demo --mode offline --output reports/offline")
display(Markdown(f"Показано **{len(traces)}** сохранённых историй."))


Выбран **live**: найден полный завершённый запуск с API.

Показано **3** сохранённых историй.

## Сводка запусков

Каждый результат связан с конкретной историей и режимом. Находки приведены по записям аудита, включая исторические замечания.


In [2]:
rows = []
for path, trace in traces:
    initial = trace["plans"][0] if trace["plans"] else {}
    final = trace["plans"][-1] if trace["plans"] else {}
    def drug_names(plan):
        return ", ".join(a["title"] for a in plan.get("actions", []) if a.get("category") == "medication" and a.get("status") != "stopped")
    findings = [f for audit in trace.get("audits", []) for f in audit.get("findings", [])]
    unverified = [f for audit in trace.get("audits", []) for f in audit.get("unverified_findings", [])]
    rows.append({"Пациент": trace["patient"]["id"], "Режим": trace["mode"], "Начальная терапия": drug_names(initial), "Итоговая терапия": drug_names(final), "Версий плана": len(trace["plans"]), "Подтверждённых записей": len(findings), "Неподтверждённых гипотез LLM": len(unverified), "Нужна экспертная проверка": trace.get("metadata", {}).get("requires_human_review", "не указано"), "Файл": str(path.relative_to(ROOT))})
display(pd.DataFrame(rows))


,Пациент,Режим,Начальная терапия,Итоговая терапия,Версий плана,Подтверждённых записей,Неподтверждённых гипотез LLM,Нужна экспертная проверка,Файл
0,patient_001,live,Ампициллин,Ампициллин,2,0,0,False,reports/live/patient_001.json
1,patient_002,live,Левофлоксацин,Левофлоксацин,2,0,2,True,reports/live/patient_002.json
2,patient_003,live,Ампициллин,Левофлоксацин,2,0,2,True,reports/live/patient_003.json


## Пациенты, динамика и планы

Каждое действие показывает исходное обоснование и идентификаторы источников. Доступные показатели сравниваются по времени наблюдения. Отсутствующие данные не заменяются нулями.


In [3]:
for path, trace in traces:
    patient = trace["patient"]
    display(Markdown(f"### {patient['id']} · {trace['mode']}\n\n{patient['age']} лет; {patient['diagnosis']}. Причина госпитализации: {patient['hospitalization_reason']}"))
    display(Markdown("**Аллергии:** " + ("неизвестны" if patient.get("allergies") is None else ", ".join(patient["allergies"]) or "нет в сценарии")))
    display(Markdown("**Сопутствующие заболевания:** " + (", ".join(patient.get("comorbidities", [])) or "нет в сценарии")))
    observations = patient["observations"] + [o for c in trace.get("checkpoints", []) for o in c["patient"]["observations"]]
    frame = pd.DataFrame(observations).drop_duplicates(["name", "unit", "day"], keep="last")
    display(frame.pivot_table(index=["name", "unit"], columns="day", values="value", aggfunc="last"))
    for checkpoint in trace.get("checkpoints", []):
        display(Markdown(f"**День {checkpoint['day']}:** " + "; ".join(e["description"] for e in checkpoint.get("events", []))))
        if checkpoint.get("execution_log"):
            display(Markdown("**Журнал действий симуляции:** относительный порядок задан полем sequence; это синтетические события."))
            display(pd.DataFrame(checkpoint["execution_log"]))
        display(Markdown("Допущения: " + "; ".join(checkpoint.get("assumptions", []))))
    for plan in trace["plans"]:
        display(Markdown(f"**План {plan['version']} · день {plan['day']} · {plan['origin']}**\n\n{plan['rationale']}"))
        display(pd.DataFrame([{"Действие": a["title"], "Доза": a.get("dose"), "Частота": a.get("frequency"), "Статус": a["status"], "Обоснование": a["indication"], "Источники": ", ".join(a["evidence_ids"])} for a in plan["actions"]]))
        if plan.get("missing_data"):
            display(Markdown("Неизвестные данные: " + "; ".join(plan["missing_data"])))


### patient_001 · live

52 лет; Внебольничная пневмония. Причина госпитализации: Синтетический случай: госпитализация для наблюдения из-за выраженной слабости и невозможности организовать амбулаторное наблюдение.

**Аллергии:** нет в сценарии

**Сопутствующие заболевания:** нет в сценарии

,day,0,3
name,unit,,
alt,U/L,21.0,NaN
ast,U/L,23.0,NaN
chest_xray_infiltrate,boolean,1.0,NaN
creatinine,µmol/L,78.0,NaN
crp,mg/L,128.0,42.00
egfr,mL/min/1.73m²,99.0,NaN
heart_rate,/min,104.0,84.00
potassium,mmol/L,4.2,NaN
qtc_ms,ms,420.0,NaN


**День 3:** 

...

,Действие,Доза,Частота,Статус,Обоснование,Источники
0,Ампициллин,"2,0 г",каждые 6 ч,active,Выбран ампициллин в/в как препарат первого ряд...,"cap-empiric, cap-parenteral, cap-beta-allergy,..."
1,Микроскопия и посев мокроты с чувствительностью,None,None,active,Микроскопия и посев мокроты показаны до начала...,"cap-sputum-culture, cap-sputum-before-antibiot..."
2,Общий (клинический) анализ крови,None,None,active,Общий анализ крови необходим для оценки тяжест...,cap-cbc
3,Биохимический анализ крови,None,None,active,Биохимический анализ крови необходим для оценк...,cap-biochemistry
4,С-реактивный белок,None,None,active,С-реактивный белок (СРБ) используется для оцен...,"cap-crp, cap-crp-dynamics"
5,ЭКГ в стандартных отведениях,None,None,active,ЭКГ в стандартных отведениях показана для искл...,cap-ecg
6,Пульсоксиметрия,None,None,active,Пульсоксиметрия необходима для оценки уровня к...,cap-pulse-oximetry
7,Контроль жизненных показателей,None,None,active,"Контроль жизненных показателей (температура, Ч...",cap-vitals
8,Достаточное питьё и ограничение чрезмерной наг...,None,None,active,Достаточное питье и ограничение чрезмерной физ...,cap-hydration
9,Оценка эффективности и безопасности через 48–7...,None,None,active,Оценка эффективности терапии через 48-72 часа ...,"cap-reassess-72h, cap-treatment-failure, cap-c..."


Неизвестные данные: Недостаточно данных о посеве мокроты до начала антибиотикотерапии, так как пациент уже госпитализирован и возможно начало антибиотикотерапии до сбора мокроты.

**План 2 · день 3 · llm**

Пациент 49 лет с нетяжелой внебольничной пневмонией, госпитализированный по социальным показаниям. На основании результатов посева мокроты, выявившего резистентность Haemophilus influenzae к ампициллину, и сохраняющейся лихорадки, а также отсутствия снижения уровня СРБ через 72 часа, необходимо скорректировать антибиотикотерапию. Левофлоксацин в/в выбран как препарат с доказанной эффективностью против H. influenzae и отсутствием резистентности в данном случае. Левофлоксацин активен против основных возбудителей пневмонии и имеет хорошую переносимость, что соответствует рекомендациям для нетяжелой ВП с учетом чувствительности выявленного возбудителя. Также сохраняются все необходимые обследования и мониторинговые процедуры для оценки эффективности терапии и состояния пациента.

,Действие,Доза,Частота,Статус,Обоснование,Источники
0,Левофлоксацин,"0,5 г",каждые 12 ч,active,Левофлоксацин в/в выбран в связи с резистентно...,"cap-beta-allergy, cap-inpatient-table, cap-flu..."
1,Общий (клинический) анализ крови,None,None,active,Общий анализ крови необходим для мониторинга в...,cap-cbc
2,Биохимический анализ крови,None,None,active,Биохимический анализ крови необходим для оценк...,cap-biochemistry
3,С-реактивный белок,None,None,active,С-реактивный белок используется для динамическ...,"cap-crp, cap-crp-dynamics"
4,ЭКГ в стандартных отведениях,None,None,active,ЭКГ в стандартных отведениях показана для искл...,cap-ecg
5,Микроскопия и посев мокроты с чувствительностью,None,None,active,Микроскопия и посев мокроты необходимы для под...,"cap-sputum-culture, cap-sputum-before-antibiot..."
6,Пульсоксиметрия,None,None,active,Пульсоксиметрия необходима для оценки уровня к...,cap-pulse-oximetry
7,Контроль жизненных показателей,None,None,active,Контроль жизненных показателей необходим для м...,cap-vitals
8,Достаточное питьё и ограничение чрезмерной наг...,None,None,active,Достаточное питье и ограничение физической наг...,cap-hydration
9,Оценка эффективности и безопасности через 48–7...,None,None,active,Оценка эффективности терапии через 48-72 часа ...,"cap-reassess-72h, cap-treatment-failure, cap-c..."


Неизвестные данные: Недостаточно данных о наличии факторов риска инфицирования полирезистентными возбудителями, которые могли бы повлиять на выбор антибиотика, кроме как резистентность к ампициллину.

## Независимый аудит

Находки правил и LLM имеют отдельную маркировку. Подтверждение связывает замечание с фактом, версией плана, действием и источником. Остальные предположения модели сохраняются отдельно для экспертной оценки. Исправленные находки остаются в истории.

Проверка покрывает заданный каталог и перечисленные правила; произвольные утверждения за их пределами автоматически не подтверждаются.


In [4]:
for path, trace in traces:
    display(Markdown(f"### {trace['patient']['id']} · {trace['mode']}"))
    if trace.get("metadata", {}).get("requires_human_review"):
        display(Markdown("**Требуется экспертная проверка.** Неподтверждённые гипотезы и недостаточные данные не считаются чистым аудитом."))
    for number, audit in enumerate(trace.get("audits", []), 1):
        display(Markdown(f"**Аудит {number}. LLM-проверка: {'да' if audit.get('llm_reviewed') else 'нет'}.**"))
        if audit.get("llm_summary"):
            display(Markdown(audit["llm_summary"]))
        records = [{"Код": f["code"], "Уровень": f["severity"], "Статус": f["status"], "Происхождение": f["origin"], "Версия": f["plan_version"], "Находка": f["message"], "Доказательство": f["patient_evidence"], "Исправление": f["recommendation"], "Источники": ", ".join(f["evidence_ids"])} for f in audit.get("findings", [])]
        display(pd.DataFrame(records)) if records else display(Markdown("Подтверждённых нарушений в покрытых проверках не найдено."))
        if audit.get("unverified_findings"):
            display(Markdown("**Неподтверждённые предположения LLM — отдельная экспертная оценка**"))
            display(pd.DataFrame([{"Код": f["code"], "Версия": f["plan_version"], "Гипотеза": f["message"], "Заявленное основание": f["patient_evidence"], "Источники": ", ".join(f["evidence_ids"])} for f in audit["unverified_findings"]]))
            if audit.get("llm_validation"):
                display(pd.DataFrame(audit["llm_validation"]))
        if audit.get("llm_raw_summary"):
            display(Markdown("**Необработанное заключение LLM (может содержать неподтверждённые утверждения):**\n\n" + audit["llm_raw_summary"]))
        display(Markdown("Ограничения: " + "; ".join(audit.get("limitations", []))))


### patient_001 · live

**Аудит 1. LLM-проверка: нет.**

Подтверждённых нарушений в покрытых проверках не найдено.

Ограничения: Проверяется ограниченный каталог выбранного сценария; отсутствие находок не доказывает клиническую безопасность.; Любой непустой список сопутствующих заболеваний находится за пределами текущих демонстрационных случаев и требует ручной оценки; это не правило о противопоказании антибиотиков при любых заболеваниях.; Связь назначения с источником проверяется по вручную подтверждённому каталогу, а не универсальным доказательством логического следования текста.; Проверка доз покрывает только заранее заданные режимы. eGFR>=90 и АЛТ/АСТ<=40 — консервативные границы демонстрационного набора, не универсальные противопоказания или полная оценка функции органов.; Правило взаимодействий явно покрывает левофлоксацин+амиодарон; полной базы лекарственных взаимодействий нет.; QTc<450 и калий>=3.5 — границы синтетического входного набора; они не доказывают отсутствие риска и не являются полной клинической стратификацией.; КР, инструкция препарата и политики демонстрации имеют отдельные идентификаторы и разную область применимости.; Наблюдения датируются с точностью до дня. При наличии синтетического execution_log его sequence отражает относительный порядок забора мокроты и начала препаратов внутри дня; без журнала этот порядок неизвестен.; Наличие отдельного показателя — ограниченное свидетельство исследования; полнота всех полей анализа и качество образца здесь не валидируются.; Historical отмечает повторяющуюся проблему старой версии; resolved сохраняет исправленную историческую находку. Только open описывает текущую нерешённую проблему.

**Аудит 2. LLM-проверка: да.**

...

**Неподтверждённые предположения LLM — отдельная экспертная оценка**

,Код,Версия,Гипотеза,Заявленное основание,Источники
0,known_culture_resistance,2,На момент принятия решения о назначении левофл...,Haemophilus influenzae с устойчивостью к ампиц...,"cap-culture-interpretation, cap-sputum-culture"
1,required_sputum_culture_missing,1,В версии 1 отсутствует подтверждение сбора мок...,Сбор мокроты зарегистрирован в журнале исполне...,cap-sputum-before-antibiotics


,index,code,plan_version,action_id,proposed,decision,accepted,reason,coverage_limitation
0,0,known_culture_resistance,2,v2:levofloxacin_iv,"{'code': 'known_culture_resistance', 'severity...",unverified,False,"Нет фактической находки с теми же code, plan_v...",Проверка предложений LLM ограничена реализован...
1,1,required_sputum_culture_missing,1,None,"{'code': 'required_sputum_culture_missing', 's...",unverified,False,"Нет фактической находки с теми же code, plan_v...",Проверка предложений LLM ограничена реализован...


**Необработанное заключение LLM (может содержать неподтверждённые утверждения):**

В версии 1 план соответствовал источникам и данным пациента, за исключением отсутствия подтверждения сбора мокроты до начала антибиотикотерапии. В версии 2 выявлено нарушение из-за известной резистентности Haemophilus influenzae к ампициллину на момент принятия решения о назначении левофлоксацина. Также отмечается отсутствие данных о согласии на процедуры и предварительной оценке функции почек для возможных исследований с контрастом, хотя сами процедуры не выполнялись.

Ограничения: Проверяется ограниченный каталог выбранного сценария; отсутствие находок не доказывает клиническую безопасность.; Любой непустой список сопутствующих заболеваний находится за пределами текущих демонстрационных случаев и требует ручной оценки; это не правило о противопоказании антибиотиков при любых заболеваниях.; Связь назначения с источником проверяется по вручную подтверждённому каталогу, а не универсальным доказательством логического следования текста.; Проверка доз покрывает только заранее заданные режимы. eGFR>=90 и АЛТ/АСТ<=40 — консервативные границы демонстрационного набора, не универсальные противопоказания или полная оценка функции органов.; Правило взаимодействий явно покрывает левофлоксацин+амиодарон; полной базы лекарственных взаимодействий нет.; QTc<450 и калий>=3.5 — границы синтетического входного набора; они не доказывают отсутствие риска и не являются полной клинической стратификацией.; КР, инструкция препарата и политики демонстрации имеют отдельные идентификаторы и разную область применимости.; Наблюдения датируются с точностью до дня. При наличии синтетического execution_log его sequence отражает относительный порядок забора мокроты и начала препаратов внутри дня; без журнала этот порядок неизвестен.; Наличие отдельного показателя — ограниченное свидетельство исследования; полнота всех полей анализа и качество образца здесь не валидируются.; Historical отмечает повторяющуюся проблему старой версии; resolved сохраняет исправленную историческую находку. Только open описывает текущую нерешённую проблему.; Отсутствие данных о согласии на процедуры и предварительной оценке функции почек для возможных исследований с контрастом не подтверждает нарушений, так как сами процедуры не выполнялись.; Недостаточно данных о факторах риска инфицирования полирезистентными возбудителями, кроме резистентности к ампициллину.; Недостаточно данных о наличии аллергий на бета-лактамные антибиотики и фторхинолоны, что не позволяет проверить соответствие требований каталога.; Проверка предложений LLM ограничена реализованными фактическими правилами. Утверждения вне этого набора сохраняются как непроверенные и требуют эксперта. Совпадение идентификаторов источников не доказывает произвольное смысловое утверждение; принятые находки используют каноническую формулировку проверенного правила.

## Проверяемые источники

Ниже приведены использованные локальные фрагменты. Номер страницы и раздел относятся к зафиксированной редакции документа; технический ID не заменяет номер пункта КР.


In [5]:
chunks = []
for name in ["document_chunks.json", "chunks.json", "chunks.jsonl", "guideline_chunks.json", "guideline_chunks.jsonl", "supplemental_chunks.json", "policies.json"]:
    path = ROOT / "data" / name
    if path.exists():
        data = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()] if path.suffix == ".jsonl" else load(path)
        records = data.get("chunks", data.get("documents", list(data.values()))) if isinstance(data, dict) else data
        role = "Автоматический фрагмент PDF" if name == "document_chunks.json" else "Опорная выдержка для проверки" if name == "chunks.json" else "Дополнительный источник"
        chunks.extend({**item, "display_role": role} for item in records)
used = {e for _, t in traces for p in t["plans"] for a in p["actions"] for e in a["evidence_ids"]}
used |= {e for _, t in traces for audit in t.get("audits", []) for f in audit["findings"] for e in f["evidence_ids"]}
used |= {e for _, t in traces for audit in t.get("audits", []) for f in audit.get("unverified_findings", []) for e in f["evidence_ids"]}
for chunk in chunks:
    identifier = chunk.get("id", chunk.get("chunk_id"))
    if identifier in used:
        page = chunk.get("page", chunk.get("pdf_page", chunk.get("pages", "—")))
        section = chunk.get("section", chunk.get("section_title", "—"))
        url = chunk.get("source_url", chunk.get("url")) or chunk.get("source", {}).get("url")
        if url and isinstance(page, int) and ".pdf" in url.lower():
            url = url.split("#", 1)[0] + f"#page={page}"
        link = f"\n\n[Открыть источник]({url})" if url else ""
        display(Markdown(f"**{identifier} · {chunk['display_role']} · {section} · страница {page or '—'}**\n\n{chunk.get('text', chunk.get('content', ''))}{link}"))
missing = used - {c.get("id", c.get("chunk_id")) for c in chunks}
if missing:
    display(Markdown("**Не найдены локальные фрагменты:** " + ", ".join(sorted(missing))))


**cap-vitals · Опорная выдержка для проверки · 2.2 · страница 13**

У всех пациентов с ВП рекомендуется провести общий осмотр, измерить показатели жизненно-важных функций(частотой дыхательных движений (ЧДД), частота сердечных сокращений (ЧСС), артериальное давление (АД),температура тела) и выполнить детальное обследование грудной клетки как неотъемлемых компонентов установлениядиагноза, оценки тяжести ВП и прогноза [1].

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=13)

**cap-pulse-oximetry · Опорная выдержка для проверки · 2.2 · страница 13**

Всем пациентам с ВП рекомендуется пульсоксиметрия с измерением SpO для выявления ДН и оценки выраженностигипоксемии [2, 93].

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=13)

**cap-cbc · Опорная выдержка для проверки · 2.3 · страница 14**

Всем пациентам с ВП рекомендуется выполнение общего (клинического) анализа крови с определением уровняэритроцитов, гематокрита, лейкоцитов, тромбоцитов, лейкоцитарной формулы с целью установления диагноза, оценкитяжести ВП и прогноза [1].

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=14)

**cap-biochemistry · Опорная выдержка для проверки · 2.3 · страница 14**

Всем госпитализированным пациентам с ВП рекомендуется выполнить анализ крови биохимическийобщетерапевтический (мочевина, креатинин, общий билирубин, глюкоза, альбумин, исследование уровня натрия, калия,хлоридов, определение активности аспартатаминотрансферазы, активности аланинамино-трансферазы в крови) с цельюопределения тяжести ВП и прогноза, выявления декомпенсации сопутствующей патологии, назначения и коррекциифармакотерапии [1].

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=14)

**cap-crp · Опорная выдержка для проверки · 2.3 · страница 14**

Всем госпитализированным пациентам с ВП рекомендуется исследование уровня
С-реактивного белка (СРБ) в сыворотке крови с целью установления диагноза, оценки тяжести ВП и прогноза [50, 259].

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=14)

...

**cap-qt · Опорная выдержка для проверки · Приложение А3, препараты других групп · страница 63**

Среди тетрациклинов наиболее приемлемым, учитывая фармакокинетические особенности, переносимость и удобствоприменения является доксициклин**. Он характеризуется хорошей активностью в отношении M. pneumoniae, C.pneumoniae и низким уровнем вторичной резистентности H. influenzae в РФ. Высокая частота выделениятетрациклинорезистентных штаммов S. pneumoniae в России не позволяет рассматривать его в качестве препарата выборадля эмпирической терапии ВП. Для госпитализированных пациентов при наличии противопоказаний к назначению b-лактамов и отсутствии других терапевтических опции может рассматриваться тигециклин; он же, учитывая наличиеактивности против L. pneumophila, M. pneumoniae, C. pneumoniae, может назначаться вместе с b-лактамными АБ приТВП и удлиненном интервале QTc, т.к. в данном случае применение макролидов и фторхинолонов противопоказано.

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=63)

**cap-dose-scope · Опорная выдержка для проверки · Приложение А3, режимы дозирования АМП · страница 64**

Режимы дозирования АМП (нормальная функция печени и почек)

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=64)

**cap-dose-ampicillin · Опорная выдержка для проверки · Приложение А3, режимы дозирования АМП · страница 64**

Ампициллин** 2,0 г в/в, в/м каждые 6 ч

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=64)

**cap-dose-levofloxacin · Опорная выдержка для проверки · Приложение А3, режимы дозирования АМП · страница 64**

Левофлоксацин** 0,5 г каждые 12 ч внутрь или в/в или 0,75 г каждые 24 ч внутрь

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=64)

**cap-hydration · Опорная выдержка для проверки · Приложение В, информация для пациентов · страница 68**

При пневмонии рекомендуется также временно ограничить чрезмерную физическую нагрузку и пить достаточноеколичество жидкости.

[Открыть источник](https://spulmo.ru/upload/KR-vnebolnichnaya-pnevmoniya-u-vzroslyh-2024.pdf#page=68)

## Проверочные прогоны

Внесённые дефекты проверяют обнаружение конкретных ошибок на копиях эталонных планов. Они отделены от трёх основных случаев. Результаты характеризуют этот набор тестов.


In [6]:
evaluation_files = sorted(path for path in (ROOT / "reports").rglob("evaluation.json") if "development" not in path.relative_to(ROOT / "reports").parts)
if not evaluation_files:
    display(Markdown("Отчёт evaluation.json отсутствует; результаты оценивания здесь не заявляются."))
for path in evaluation_files:
    display(Markdown(f"**{path.relative_to(ROOT)}**"))
    display(load(path))


**reports/evaluation/evaluation.json**

{'mode': 'offline',
 'fault_injection': True,
 'note': 'Базовые планы для мутаций — явно обозначенные офлайн-эталоны. LLM-аудит выполняется независимо в live-примерах дозы и процедурных условий. Это малый демонстрационный набор, не оценка клинической надежности.',
 'rule_detection': {'detected': 9,
  'total': 9,
  'cases': [{'id': 'wrong_dose',
    'expected': 'catalog_parameter_mismatch',
    'detected': True,
    'found_codes': ['catalog_parameter_mismatch']},
   {'id': 'irrelevant_citation',
    'expected': 'citation_support_incomplete',
    'detected': True,
    'found_codes': ['citation_support_incomplete', 'citation_unrelated']},
   {'id': 'invented_citation',
    'expected': 'citation_missing',
    'detected': True,
    'found_codes': ['citation_missing',
     'citation_support_incomplete',
     'citation_unrelated']},
   {'id': 'ignored_allergy',
    'expected': 'beta_lactam_allergy',
    'detected': True,
    'found_codes': ['beta_lactam_allergy']},
   {'id': 'drug_interaction

**reports/evaluation_live/evaluation.json**

{'mode': 'live',
 'fault_injection': True,
 'note': 'Базовые планы для мутаций — явно обозначенные офлайн-эталоны. LLM-аудит выполняется независимо в live-примерах дозы и процедурных условий. Это малый демонстрационный набор, не оценка клинической надежности.',
 'rule_detection': {'detected': 9,
  'total': 9,
  'cases': [{'id': 'wrong_dose',
    'expected': 'catalog_parameter_mismatch',
    'detected': True,
    'found_codes': ['catalog_parameter_mismatch']},
   {'id': 'irrelevant_citation',
    'expected': 'citation_support_incomplete',
    'detected': True,
    'found_codes': ['citation_support_incomplete', 'citation_unrelated']},
   {'id': 'invented_citation',
    'expected': 'citation_missing',
    'detected': True,
    'found_codes': ['citation_missing',
     'citation_support_incomplete',
     'citation_unrelated']},
   {'id': 'ignored_allergy',
    'expected': 'beta_lactam_allergy',
    'detected': True,
    'found_codes': ['beta_lactam_allergy']},
   {'id': 'drug_interaction',
